# E8d: dinámica de la actividad

Referencia: dos XGBoost con MPUD (uno para `y_f`, otro para `y_s`) con las 36 features podadas. Se agregan bloques de dinámica sin perfil (velocidad, aceleración, cambio relativo, centro de masa, pico y persistencia de la tendencia a 15 min, 1 h y 24 h, y cocientes entre escalas). Criterio fijado antes (`README.md`, E8d): un bloque se queda en un modelo si le suma 0.01 o más de AUC (mediana por persona dejando una persona fuera, media de 5 semillas).

Meta del modelo de producción: AUC cerca de 0.72 por patrón y, en cada persona, mejor que la persistencia.

In [1]:
import os, json
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
import numpy as np
import pandas as pd
from experiments.modeling import load, feature_cols, evaluate, persistence_score
from experiments import variants as V
from smartuse.features_dynamics import GROUPS
SEEDS = [0, 1, 2, 3, 4]
TARGETS = ["y_f", "y_s"]

def with_dynamics(name):
    base = load(name)
    dyn = pd.read_csv(f"data/processed/{name}_features_dynamics.csv")
    assert (base["t_utc"].values == dyn["t_utc"].values).all()
    return pd.concat([base, dyn.drop(columns=["user_id", "t_utc"])], axis=1)

real, mpud = with_dynamics("real"), with_dynamics("mpud")
base = V.pruned(feature_cols(load("real")))
configs = {"baseline": base, **{f"+ {g}": base + cols for g, cols in GROUPS.items()},
           "+ all dynamics": base + [c for cols in GROUPS.values() for c in cols]}
{k: len(v) for k, v in configs.items()}

{'baseline': 36,
 '+ dyn_15m': 48,
 '+ dyn_1h': 48,
 '+ dyn_24h': 48,
 '+ dyn_cross': 42,
 '+ all dynamics': 78}

## Corridas (2 modelos × 6 configuraciones × 5 semillas)

Los scores se guardan en `data/reports/e8d_scores.parquet` y se reutilizan si el notebook se vuelve a correr.

In [2]:
cache = Path("data/reports/e8d_scores.parquet")
scores = pd.read_parquet(cache) if cache.exists() else pd.DataFrame(index=real.index)
for target in TARGETS:
    for name, cols in configs.items():
        for seed in SEEDS:
            key = f"{target}|{name}|{seed}"
            if key not in scores:
                V.set_seed(seed)
                scores[key] = V.scores_for(real, mpud, cols, target)
                scores.to_parquet(cache)
rows = [{"target": t, "config": c, "seed": int(s), "auc": V.auc(real, scores[k].to_numpy(), t)}
        for k in scores for t, c, s in [k.split("|")]]
runs = pd.DataFrame(rows)
mean = runs.pivot_table(index="config", columns="target", values="auc", aggfunc="mean").loc[list(configs)]
gain = mean - mean.loc["baseline"]
pd.concat({"AUC (media)": mean, "ganancia": gain}, axis=1).round(3)

AUC (media)        ganancia       
target                 y_f    y_s      y_f    y_s
config                                           
baseline             0.693  0.689    0.000  0.000
+ dyn_15m            0.692  0.694   -0.001  0.005
+ dyn_1h             0.696  0.696    0.004  0.007
+ dyn_24h            0.701  0.674    0.009 -0.014
+ dyn_cross          0.691  0.696   -0.002  0.007
+ all dynamics       0.705  0.679    0.012 -0.010

In [3]:
runs.groupby(['config', 'target'])['auc'].agg(['min', 'max']).unstack().loc[list(configs)].round(3)

min           max       
target            y_f    y_s    y_f    y_s
config                                    
baseline        0.689  0.679  0.697  0.699
+ dyn_15m       0.685  0.681  0.702  0.703
+ dyn_1h        0.690  0.684  0.705  0.704
+ dyn_24h       0.691  0.667  0.710  0.680
+ dyn_cross     0.673  0.691  0.700  0.700
+ all dynamics  0.693  0.670  0.715  0.686

## Decisión por modelo

In [4]:
keep = {t: [g for g in GROUPS if gain.loc[f"+ {g}", t] >= 0.01] for t in TARGETS}
best = {t: max(configs, key=lambda c: mean.loc[c, t]) for t in TARGETS}
for t in TARGETS:
    print(f"{t}: keep {keep[t] or 'nothing'}; all dynamics {gain.loc['+ all dynamics', t]:+.3f}; "
          f"best config {best[t]} ({mean.loc[best[t], t]:.3f})")

y_f: keep nothing; all dynamics +0.012; best config + all dynamics (0.705)
y_s: keep nothing; all dynamics -0.010; best config + dyn_1h (0.696)


## Meta por persona

Score promedio de las 5 semillas; AUC por persona del modelo frente a la persistencia de su patrón (F₆₀ o S₆₀ de la última hora sobre el perfil).

In [5]:
persist = {"y_f": (real.f_60 > real.prof_pf).astype(float), "y_s": (real.s_60 > real.prof_ps).astype(float)}
goal = {}
for t in TARGETS:
    p_auc = evaluate(real[t], persist[t].to_numpy(), real.user_id, persistence_score(real), n_boot=10)["auc_per_person"]
    for c in ("baseline", best[t]):
        avg = np.mean([scores[f"{t}|{c}|{s}"] for s in SEEDS], axis=0)
        r = evaluate(real[t], avg, real.user_id, persistence_score(real), n_boot=200)
        diff = pd.Series(r["auc_per_person"]) - pd.Series(p_auc)
        goal[(t, c)] = {"AUC median": r["auc_median"], "CI95": "%.3f-%.3f" % tuple(r["auc_median_ci95"]),
                        "worst person": r["auc_min"], "persons beating persistence": f"{(diff > 0).sum()}/{diff.notna().sum()}",
                        "smallest margin": diff.min()}
pd.DataFrame(goal).T.round(3)

AUC median         CI95 worst person  \
y_f baseline          0.69822  0.665-0.724      0.48917   
    + all dynamics   0.713129  0.676-0.729     0.487118   
y_s baseline         0.686753  0.648-0.716     0.575037   
    + dyn_1h         0.700026  0.654-0.723     0.573197   

                   persons beating persistence smallest margin  
y_f baseline                             17/17        0.011001  
    + all dynamics                       17/17        0.008949  
y_s baseline                             16/16        0.041427  
    + dyn_1h                             16/16        0.046457

## Guardar

In [6]:
out = {"mean": mean.to_dict(), "gain": gain.to_dict(), "runs": runs.to_dict("records"), "keep": keep,
       "best": best, "goal": {f"{t}|{c}": v for (t, c), v in goal.items()}}
Path("data/reports/e8d_dynamics.json").write_text(json.dumps(out, indent=2, default=float))
print("-> data/reports/e8d_dynamics.json")

-> data/reports/e8d_dynamics.json
